In [1]:
# imports needed for this project
import os
import time
import pandas as pd
import spacy
from spacy.matcher import Matcher
from classes import AccidentData, MetaData
from scrapers import read_links_from_file, scrape_metadata, write_metadata_to_csv
import re
from transformers import pipeline


c:\Users\hucu\OneDrive\Pulpit\NLP-project\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
import spacy
from spacy.matcher import Matcher

nlp = spacy.load('en_core_web_sm')


def setup_fatalities_matcher(nlp):
    matcher = Matcher(nlp.vocab)
    
    # Pattern for "FirstName LastName, age"
    pattern1 = [
        {"POS": "PROPN"},           
        {"IS_PUNCT": True, "OP": "?"},  
        {"POS": "PROPN"},           
        {"IS_PUNCT": True, "TEXT": ","}, 
        {"POS": "NUM"}              
    ]

    # Pattern for "FirstName, age"
    pattern2 = [
        {"POS": "PROPN"},           
        {"IS_PUNCT": True, "TEXT": ","}, 
        {"POS": "NUM"}              
    ]

    # Add both patterns to the matcher
    matcher.add("NameAgePattern", [pattern1, pattern2])

    return matcher


def remove_shorter_duplicates(names):
    sorted_names = sorted(names, key=len, reverse=True)  
    final_names = []

    for name in sorted_names:
        if not any(name in existing_name for existing_name in final_names if existing_name != name):
            final_names.append(name)

    return final_names

def extract_fatalities(doc, matcher):
    death_related_terms = ['killed', 'deceased', 'succumbed', 'died', 'fatalities']
    fatality_context = False
    potential_victims = []
    actual_fatalities = []

    for sent in doc.sents:
        matches = matcher(sent)
        for match_id, start, end in matches:
            potential_victims.append(sent[start:end-2].text)  

        if any(death_term in sent.text.lower() for death_term in death_related_terms):
            fatality_context = True
            actual_fatalities.extend(potential_victims)
            potential_victims = []  
        else:
            fatality_context = False

    return remove_shorter_duplicates(actual_fatalities)

# Test with text example
text = """
The deceased were identified as Mostafa, 35, his wife Josna, 28, and their grandmother Johra Beowa, 75, of Jigamarir Ghat area in Kurigram Sadar.

Mahfuzar Rahman, officer-in-charge, Kurigram Sadar Police Station said a Rangpur-bound minibus hit a battery-run auto-rickshaw beside Kathalbari College around noon.

One person was killed on the spot. The other two succumbed to their injuries at the sadar hospital, said Civil Surgeon Dr SM Aminul Islam.

Auto-rickshaw driver Manik, 35, was shifted to Rangpur Medical College Hospital as his condition deteriorated. The other two injured – Jahedul Islam, 45 and Mim – were being treated at the sadar hospital.
"""
doc = nlp(text)
fatalities = extract_fatalities(doc, setup_fatalities_matcher(nlp))

print("Fatalities:", fatalities)  

Fatalities: ['Johra Beowa', 'Mostafa', 'Josna']


In [ ]:
def extract_details(meta_data_list):
    processed_data = []

    for text in meta_data_list:
        doc = nlp(str(text.raw_text))

        # # location
        # locations = [ent.text for ent in doc.ents if ent.label_ == 'GPE']
        # locations = clean_locations(locations)
        # exact_locations = extract_location(doc)
        # exact_locations = remove_duplicates_simple(exact_locations)
        
        # #Extract date, find day of the week etc.
        # date_info = [ent.text for ent in doc.ents if ent.label_ == 'DATE' or ent.label_ == 'TIME']
        # date_info = clean_dates(date_info)

        # # vehicles
        # matches = matcher_vechicles(doc)
        # vehicles = [doc[start:end].text for match_id, start, end in matches]
        # vehicles = map_vehicle_names(clean_and_deduplicate(vehicles))

        # casualties 
        casualties = extract_fatalities(doc, setup_fatalities_matcher(nlp))
        print(casualties)

        # number of injured persons
        #injured = extract_casualties(doc)

        # reason for the accident
        #reason = extract_reason_hf(text.raw_text)

        # sequence of actions
        #actions = extract_sequence_of_actions_spacy(doc)

        # AccidentData object
        

    return processed_data

In [ ]:
def process_and_save(meta_data_list, output_file):
    processed_data = extract_details(meta_data_list)
    df = pd.DataFrame(processed_data)
    df.to_csv(output_file, sep=';', index=False)

links_to_scrape = read_links_from_file("test.txt")
meta_data_list = scrape_metadata(links_to_scrape)
write_metadata_to_csv(meta_data_list, "test0")
process_and_save(meta_data_list, 'processed_events.csv')

Scrapping [https://www.unb.com.bd/category/Bangladesh/man-killed-in-kushtia-road-crash/4366]
Scrapping [https://www.unb.com.bd/category/Bangladesh/truck-ambulance-collision-leaves-one-dead-in-natore/3597]
